# Phenology Composite → SAM Segmentation
### SEPAL | Sentinel-2 | SAM2 automatic mask generation

**Pipeline:**
1. Load a 12-band monthly NDVI stack exported from SEPAL
2. Build a phenological RGB composite — two modes:
   - `tri_temporal`: early / peak / late season NDVI → R / G / B
   - `pca`: IncrementalPCA on all months → PC2 / PC3 / temporal CV
3. Reproject to UTM if needed
4. Run SAM2 on the composite (full scene or tiled)
5. Vectorise the mask to a GeoPackage
6. Plot and inspect results

**Requirements:** SEPAL instance (m2+), `(venv) SamGeo` kernel

```bash
# Run once in SEPAL terminal to create the kernel:
python3 -m venv ~/venvs/samgeo-env
source ~/venvs/samgeo-env/bin/activate

pip install "segment-geospatial[samgeo2]" ipykernel \
            rasterio geopandas matplotlib scikit-image \
            scikit-learn scipy psutil pyproj

python -m ipykernel install --user \
       --name=samgeo-env --display-name='(venv) SamGeo'

deactivate
```

---
## Step 1 — Imports

In [1]:
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['OPENCV_LOG_LEVEL'] = 'ERROR'
import logging
logging.getLogger('rasterio').setLevel(logging.ERROR)

import os, gc, time
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.features import shapes as rio_shapes
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from skimage import exposure
import psutil
import torch

def ram_gb():
    return psutil.virtual_memory().available / 1e9

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
print(f'RAM     : {ram_gb():.1f} GB available')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE}')

PyTorch : 2.11.0+cu130
CUDA    : True
RAM     : 12.8 GB available
Device  : cuda


---
## Step 2 — Configuration

Set all parameters here. Nothing else needs editing.

**Input:** A single GeoTIFF with N bands (one per month), exported from a SEPAL
Time Series recipe. All tiles should be in one folder — the notebook merges them.

**Band selection:** Set `EARLY_MONTH`, `PEAK_MONTH`, `LATE_MONTH` to 1-based month
numbers, or `None` for automatic selection based on mean NDVI.

In [2]:
# ── Run mode ──────────────────────────────────────────────────────────────────
CLEAN_RUN = True

# ── Input ─────────────────────────────────────────────────────────────────────
NDVI_TILE_DIR = os.path.expanduser('~/downloads/Phenology_2026-04-28_13-54-52')

MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

# ── Composite mode ────────────────────────────────────────────────────────────
# 'tri_temporal' : R=EARLY, G=PEAK, B=LATE  (fast, 3 bands)
# 'pca'          : PCA on all months -> R=PC2, G=PC3, B=temporal CV
COMPOSITE_MODE = 'pca'

# ── Tri-temporal band selection (tri_temporal mode only) ──────────────────────
EARLY_MONTH = 4
PEAK_MONTH  = 9
LATE_MONTH  = 11

# ── PCA settings (pca mode only) ──────────────────────────────────────────────
PCA_R_COMPONENT  = 1      # 0-indexed: PC2 = first residual after dominant gradient
PCA_G_COMPONENT  = 2      # PC3 = subtle field-level differences
B_CHANNEL_METHOD = 'cv'   # 'cv' = temporal coefficient of variation
                           # 'local_contrast' = high-pass spatial filter
# Focus on growing season — skip dry months where all fields look the same
SELECTED_MONTHS  = [4, 5, 6, 7, 8, 9, 10, 11]  # Apr-Nov

# ── Reprojection (Step 4b) ────────────────────────────────────────────────────
TARGET_CRS   = 'auto'
TARGET_RES_M = None

# ── Stretch method ────────────────────────────────────────────────────────────
# 'percentile'  linear 2-98th percentile (fast, predictable)
# 'equalize'    global histogram equalisation (maximum contrast)
# 'adaptive'    CLAHE local contrast (best for SAM, slowest)
# Use percentile for PCA — scores are already zero-centred,
# percentile stretch is more stable than adaptive on PCA output
STRETCH_METHOD = 'equalize'

# ── SAM parameters ────────────────────────────────────────────────────────────
POINTS_PER_SIDE               = 128   # reduced from 128 — PCA composite has
                                      # sharper edges so fewer points needed
PRED_IOU_THRESH               = 0.55
STABILITY_THRESH              = 0.75
MIN_MASK_AREA                 = 10
CROP_N_LAYERS                 = 1
CROP_N_POINTS_DOWNSCALE_FACTOR = 1

# ── Tiling ────────────────────────────────────────────────────────────────────
USE_TILING    = True
SAM_TILE_SIZE = 256
TILE_OVERLAP  = 128

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR    = os.path.expanduser('~/downloads/phenology_sam')
os.makedirs(OUTPUT_DIR, exist_ok=True)

MERGED_TIF    = os.path.join(OUTPUT_DIR, 'merged_bands.tif')
PHENO_RGB_TIF = os.path.join(OUTPUT_DIR, 'phenology_rgb.tif')
SAM_MASKS_TIF = os.path.join(OUTPUT_DIR, 'sam_masks.tif')
PRED_DIR      = os.path.join(OUTPUT_DIR, 'sam_tiles')
SAM_VECTOR    = os.path.join(OUTPUT_DIR, 'sam_segments.gpkg')
MEMMAP_PATH   = os.path.join(OUTPUT_DIR, '_sam_input.dat')

SCALE_FACTOR  = 10000.0

print('Configuration set.')
print(f'  Input folder    : {NDVI_TILE_DIR}')
print(f'  Composite mode  : {COMPOSITE_MODE}')
print(f'  Selected months : {SELECTED_MONTHS}')
print(f'  PCA components  : R=PC{PCA_R_COMPONENT+1}  G=PC{PCA_G_COMPONENT+1}  B={B_CHANNEL_METHOD}')
print(f'  Stretch         : {STRETCH_METHOD}')
print(f'  Output dir      : {OUTPUT_DIR}')
print(f'  Tiling          : {USE_TILING}  ({SAM_TILE_SIZE}px tiles, {TILE_OVERLAP}px overlap)')

Configuration set.
  Input folder    : /home/dguerrero/downloads/Phenology_2026-04-28_13-54-52
  Composite mode  : pca
  Selected months : [4, 5, 6, 7, 8, 9, 10, 11]
  PCA components  : R=PC2  G=PC3  B=cv
  Stretch         : equalize
  Output dir      : /home/dguerrero/downloads/phenology_sam
  Tiling          : True  (256px tiles, 128px overlap)


---
## Step 2b — Apply Clean Run

If `CLEAN_RUN = True`, deletes all previously generated outputs so every
subsequent step runs from scratch. Set `CLEAN_RUN = False` to reuse
existing outputs and skip steps that are already complete.

In [3]:
import glob

# Files and folders that each step produces
# Ordered so that dependent outputs are removed before their inputs
OUTPUTS_TO_CLEAN = [
    # Step 9 — vector output
    SAM_VECTOR,
    # Step 7 — memmap input array
    MEMMAP_PATH,
    # Step 8 — tile predictions folder
    PRED_DIR,
    # Step 7 — SAM mask
    SAM_MASKS_TIF,
    # Step 5 — composite
    PHENO_RGB_TIF,
    # Step 4b — reprojected merged file (wildcard — any CRS suffix)
    # (base merged file without suffix is handled by MERGED_TIF below)
    *glob.glob(os.path.splitext(os.path.join(
        OUTPUT_DIR, 'merged_*.tif'
    ))[0] + '_EPSG*.tif'),
    # Step 4 — merged band file
    *glob.glob(os.path.join(OUTPUT_DIR, 'merged_*.tif')),
]

if CLEAN_RUN:
    print('CLEAN_RUN = True — removing existing outputs...')
    import shutil
    removed = []
    for path in OUTPUTS_TO_CLEAN:
        if os.path.isdir(path):
            shutil.rmtree(path)
            removed.append(f'  DIR  {path}')
        elif os.path.isfile(path):
            os.remove(path)
            removed.append(f'  FILE {path}')
    if removed:
        for r in removed:
            print(r)
        print(f'Removed {len(removed)} item(s).')
    else:
        print('Nothing to remove — output directory was already clean.')
    print('\nAll subsequent steps will run from scratch.')
else:
    # Report what already exists
    print('CLEAN_RUN = False — checking existing outputs:')
    checks = [
        (glob.glob(os.path.join(OUTPUT_DIR, 'merged_*.tif')),
         'Step 4  merged bands'),
        (glob.glob(os.path.join(OUTPUT_DIR, 'merged_*EPSG*.tif')),
         'Step 4b reprojected'),
        ([PHENO_RGB_TIF],   'Step 5  composite'),
        ([SAM_MASKS_TIF],   'Step 7  SAM mask'),
        ([PRED_DIR],        'Step 7  tile folder'),
        ([SAM_VECTOR],      'Step 9  vector output'),
    ]
    for paths, label in checks:
        existing = [p for p in paths if os.path.exists(p)]
        if existing:
            names = ', '.join(os.path.basename(p) for p in existing)
            print(f'  FOUND   {label}: {names}')
        else:
            print(f'  MISSING {label}')
    print('\nSteps with existing outputs will be skipped (cached).')
    print('Set CLEAN_RUN = True in Step 2 to delete and re-run everything.')


CLEAN_RUN = True — removing existing outputs...
Nothing to remove — output directory was already clean.

All subsequent steps will run from scratch.


---
## Step 3 — Discover Tiles and Inspect Stack

Scans the input folder for `.tif` files and reads metadata from the first tile.
No full-resolution pixels loaded yet.

In [4]:
import glob

# Discover tiles
tif_files = sorted(glob.glob(os.path.join(os.path.expanduser(NDVI_TILE_DIR), '*.tif')))
print(f'Found {len(tif_files)} tile(s) in {NDVI_TILE_DIR}')
for f in tif_files[:5]: print(f'  {os.path.basename(f)}')
if len(tif_files) > 5: print(f'  ... and {len(tif_files)-5} more')

if len(tif_files) == 0:
    raise FileNotFoundError(f'No .tif files in {NDVI_TILE_DIR}. Check NDVI_TILE_DIR.')

# Metadata from first tile
with rasterio.open(tif_files[0]) as src:
    n_months    = src.count
    TILE_H      = src.height
    TILE_W      = src.width
    REF_PROFILE = src.profile.copy()
    nodata_val  = src.nodata
    STACK_CRS   = src.crs
    STACK_RES   = src.res
    # Small sample for stats
    sample = src.read(
        out_shape=(n_months, min(200, TILE_H), min(200, TILE_W)),
        resampling=Resampling.average
    ).astype(np.float32)

print(f'\nBands per tile : {n_months}')
print(f'CRS            : EPSG:{STACK_CRS.to_epsg()}')
print(f'Resolution     : {STACK_RES[0]:.1f} m')
print(f'Nodata         : {nodata_val}')

# Detect scale
sv = sample[sample > 0]
NDVI_SCALE = SCALE_FACTOR if (len(sv) > 0 and np.percentile(sv, 99) > 2.0) else 1.0
print(f'Value range    : {"scaled 0-10000" if NDVI_SCALE > 1 else "float 0-1"}')

# Per-band stats
print(f'\nPer-band NDVI stats (sample):')
monthly_means = []
for i in range(n_months):
    b    = sample[i] / NDVI_SCALE
    v    = b[(b > -1) & (b < 1) & np.isfinite(b)]
    lbl  = MONTH_LABELS[i] if i < len(MONTH_LABELS) else f'B{i+1}'
    mn   = float(v.mean()) if len(v) > 0 else np.nan
    vpct = len(v) / b.size * 100
    monthly_means.append(mn)
    warn = '  <- LOW COVERAGE' if vpct < 70 else ''
    print(f'  {i+1:2d} {lbl:<5} mean={mn:6.3f}  valid={vpct:5.1f}%{warn}')
monthly_means = np.array(monthly_means)

# Select tri-temporal bands
n_third = n_months // 3
def resolve_month(setting, fallback):
    if setting is not None:
        idx = setting - 1
        if not (0 <= idx < n_months):
            raise ValueError(f'Month {setting} out of range (stack has {n_months} bands)')
        return idx
    w = [(i, monthly_means[i]) for i in fallback if np.isfinite(monthly_means[i])]
    return max(w, key=lambda x: x[1])[0]

early_idx = resolve_month(EARLY_MONTH, list(range(0, n_third)))
peak_idx  = resolve_month(PEAK_MONTH,  list(range(n_third, 2*n_third)))
late_idx  = resolve_month(LATE_MONTH,  list(range(2*n_third, n_months)))

el = MONTH_LABELS[early_idx] if early_idx < len(MONTH_LABELS) else f'Band {early_idx+1}'
pl = MONTH_LABELS[peak_idx]  if peak_idx  < len(MONTH_LABELS) else f'Band {peak_idx+1}'
ll = MONTH_LABELS[late_idx]  if late_idx  < len(MONTH_LABELS) else f'Band {late_idx+1}'

mode = 'auto' if EARLY_MONTH is None else 'manual'
print(f'\nBand selection ({mode}):')
print(f'  R = band {early_idx+1} ({el})  mean={monthly_means[early_idx]:.3f}')
print(f'  G = band {peak_idx+1}  ({pl})  mean={monthly_means[peak_idx]:.3f}')
print(f'  B = band {late_idx+1} ({ll})  mean={monthly_means[late_idx]:.3f}')

Found 0 tile(s) in /home/dguerrero/downloads/Phenology_2026-04-28_13-54-52


FileNotFoundError: No .tif files in /home/dguerrero/downloads/Phenology_2026-04-28_13-54-52. Check NDVI_TILE_DIR.

---
## Step 4 — Merge Selected Bands from Tiles

Reads the three selected bands directly from the `.tif` tiles using
`rasterio.merge`, bypassing the VRT. Writes a single tiled GeoTIFF
with overviews for fast previews. **Cached** — skip if file exists.

In [ ]:
from rasterio.merge import merge as rio_merge

# ── Determine which bands to merge ───────────────────────────────────────────
if COMPOSITE_MODE == 'tri_temporal':
    BANDS_NEEDED = sorted(set([early_idx+1, peak_idx+1, late_idx+1]))
else:  # pca — merge all selected months
    if SELECTED_MONTHS is not None:
        BANDS_NEEDED = sorted([m for m in SELECTED_MONTHS if 1 <= m <= n_months])
    else:
        BANDS_NEEDED = list(range(1, n_months + 1))

band_str   = '_'.join(str(b) for b in BANDS_NEEDED)
MERGED_TIF = os.path.join(OUTPUT_DIR, f'merged_{band_str}_{COMPOSITE_MODE}.tif')

labels_needed = [MONTH_LABELS[b-1] if b-1 < len(MONTH_LABELS) else f'Band {b}'
                 for b in BANDS_NEEDED]
print(f'Composite mode : {COMPOSITE_MODE}')
print(f'Bands needed   : {labels_needed}')
print(f'Output         : {MERGED_TIF}')

if os.path.exists(MERGED_TIF):
    print(f'Cached — delete to re-merge.')
else:
    print(f'Merging {len(BANDS_NEEDED)} band(s) from {len(tif_files)} tile(s)...')
    t0 = time.time()

    handles = [rasterio.open(f) for f in tif_files]
    _s, mosaic_transform = rio_merge(handles, indexes=[1], nodata=nodata_val)
    full_H, full_W = _s.shape[1], _s.shape[2]
    for h in handles: h.close()
    del _s
    print(f'  Full scene: {full_W} x {full_H} px  ({full_H*full_W/1e6:.1f} M px)')

    out_profile = REF_PROFILE.copy()
    out_profile.update(
        count=len(BANDS_NEEDED), dtype=rasterio.float32,
        height=full_H, width=full_W, transform=mosaic_transform,
        compress='lzw', tiled=True, blockxsize=512, blockysize=512,
        driver='GTiff', nodata=np.nan
    )

    with rasterio.open(MERGED_TIF, 'w', **out_profile) as dst:
        for out_idx, src_band in enumerate(BANDS_NEEDED, start=1):
            lbl = MONTH_LABELS[src_band-1] if src_band-1 < len(MONTH_LABELS) else f'Band {src_band}'
            print(f'  Band {out_idx}/{len(BANDS_NEEDED)}: {lbl}...', end=' ')
            t_b = time.time()
            handles = [rasterio.open(f) for f in tif_files]
            merged, _ = rio_merge(handles, indexes=[src_band],
                                  nodata=nodata_val, method='first')
            for h in handles: h.close()
            data = merged[0].astype(np.float32) / NDVI_SCALE
            data[(data < -1) | (data > 1)] = np.nan
            dst.write(data, out_idx)
            dst.update_tags(out_idx, MONTH=lbl, SRC_BAND=src_band)
            del merged, data; gc.collect()
            print(f'{time.time()-t_b:.1f}s')

    with rasterio.open(MERGED_TIF, 'r+') as dst:
        dst.build_overviews([2,4,8,16,32], Resampling.average)
        dst.update_tags(ns='rio_overview', resampling='average')

    elapsed = time.time() - t0
    mb = os.path.getsize(MERGED_TIF) / 1e6
    print(f'Done: {elapsed:.0f}s  {mb:.0f} MB  overviews built')

# ── Remap band indices ────────────────────────────────────────────────────────
BAND_REMAP = {src_b: out_b for out_b, src_b in enumerate(BANDS_NEEDED, start=1)}

# Tri-temporal remaps (used by Step 5 in tri_temporal mode and for previews)
if COMPOSITE_MODE == 'tri_temporal':
    merged_early_idx = BAND_REMAP[early_idx+1] - 1
    merged_peak_idx  = BAND_REMAP[peak_idx+1]  - 1
    merged_late_idx  = BAND_REMAP[late_idx+1]  - 1
else:
    # In PCA mode the merged file has bands in SELECTED_MONTHS order
    # Store for preview — use first/middle/last of selected bands
    n_sel = len(BANDS_NEEDED)
    merged_early_idx = 0
    merged_peak_idx  = n_sel // 2
    merged_late_idx  = n_sel - 1

with rasterio.open(MERGED_TIF) as src:
    H, W = src.height, src.width
    REF_PROFILE = src.profile.copy()

print(f'\nMerged file : {os.path.basename(MERGED_TIF)}')
print(f'Scene size  : {W} x {H} px  ({H*W/1e6:.1f} M pixels)')
print(f'Bands       : {len(BANDS_NEEDED)}  ({labels_needed})')

---
## Step 4b — Reproject to UTM (or Equal Area)

If the merged file is in geographic coordinates (EPSG:4326, degrees), reproject
it to a metric projection before building the composite and running SAM.

**Why this matters:**
- SAM expects square pixels — geographic pixels are rectangular (wider east-west)
- Pixel size in degrees confuses any size-based calculations
- UTM gives accurate area estimates without reprojecting the output

**Projection options** (`TARGET_CRS`):
- `'auto'` — detect the correct UTM zone automatically from the scene centre
- `'EPSG:XXXXX'` — any explicit EPSG code (e.g. `'EPSG:32736'` for UTM Zone 36S)
- `'EPSG:6933'` — WGS84 Equal Earth (global equal-area, good for any location)

**Cached** — skips if the reprojected file already exists.

In [ ]:
import subprocess
from pyproj import CRS, Transformer
import math

# ── Config ────────────────────────────────────────────────────────────────────
TARGET_CRS   = 'auto'   # 'auto', 'EPSG:32736', 'EPSG:6933', etc.
TARGET_RES_M = None     # output pixel size in metres (10m = native Sentinel-2)
                        # set None to keep native resolution after reprojection

# ── Check if reprojection is needed ──────────────────────────────────────────
with rasterio.open(MERGED_TIF) as src:
    src_crs    = src.crs
    src_res    = src.res
    src_bounds = src.bounds
    src_epsg   = src_crs.to_epsg()

# rasterio CRS has no .name — get it via pyproj
from pyproj import CRS as pCRS
crs_name = pCRS.from_user_input(src_crs.to_wkt()).name
print(f'Current CRS      : EPSG:{src_epsg}  ({crs_name})')
print(f'Current res      : {src_res[0]:.8f} x {src_res[1]:.8f}')

is_geographic = pCRS.from_user_input(src_crs.to_wkt()).is_geographic
print(f'Is geographic    : {is_geographic}')

if not is_geographic and TARGET_RES_M is None:
    print('Already in a metric projection and no target resolution set — skipping.')
    REPROJECTED_TIF = MERGED_TIF
else:
    # ── Determine target CRS ─────────────────────────────────────────────────
    if TARGET_CRS == 'auto':
        # Find UTM zone from scene centre
        cx = (src_bounds.left + src_bounds.right)  / 2
        cy = (src_bounds.bottom + src_bounds.top) / 2
        if is_geographic:
            lon, lat = cx, cy
        else:
            t = Transformer.from_crs(src_crs, 'EPSG:4326', always_xy=True)
            lon, lat = t.transform(cx, cy)

        utm_zone   = int((lon + 180) / 6) + 1
        hemisphere = 'north' if lat >= 0 else 'south'
        base_epsg  = 32600 if lat >= 0 else 32700
        target_epsg = base_epsg + utm_zone
        resolved_crs = f'EPSG:{target_epsg}'
        print(f'\nAuto UTM: lon={lon:.2f} lat={lat:.2f} -> '
              f'Zone {utm_zone}{"N" if lat>=0 else "S"} ({resolved_crs})')
    else:
        resolved_crs = TARGET_CRS
        print(f'\nTarget CRS: {resolved_crs}')

    # ── Build output filename ─────────────────────────────────────────────────
    crs_tag = resolved_crs.replace(':', '').replace('/', '')
    res_tag = f'_{int(TARGET_RES_M)}m' if TARGET_RES_M else ''
    base    = os.path.splitext(MERGED_TIF)[0]
    REPROJECTED_TIF = f'{base}_{crs_tag}{res_tag}.tif'

    if os.path.exists(REPROJECTED_TIF):
        print(f'Cached: {os.path.basename(REPROJECTED_TIF)}')
        print('Delete to force re-projection.')
    else:
        print(f'Reprojecting to {resolved_crs}'
              f'{f" at {TARGET_RES_M}m" if TARGET_RES_M else ""}...')
        t0 = time.time()

        # Build gdalwarp command
        cmd = [
            'gdalwarp',
            '-t_srs', resolved_crs,
            '-r',     'bilinear',    # bilinear for continuous NDVI values
            '-co',    'COMPRESS=LZW',
            '-co',    'TILED=YES',
            '-co',    'BLOCKXSIZE=512',
            '-co',    'BLOCKYSIZE=512',
            '-co',    'BIGTIFF=IF_SAFER',
        ]
        if TARGET_RES_M is not None:
            cmd += ['-tr', str(TARGET_RES_M), str(TARGET_RES_M)]
        cmd += [MERGED_TIF, REPROJECTED_TIF]

        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print(f'gdalwarp error:\n{result.stderr}')
            raise RuntimeError('Reprojection failed.')

        elapsed = time.time() - t0
        mb = os.path.getsize(REPROJECTED_TIF) / 1e6
        print(f'Done: {elapsed:.0f}s  {mb:.0f} MB')

        # Build overviews on reprojected file
        print('Building overviews...', end=' ')
        with rasterio.open(REPROJECTED_TIF, 'r+') as dst:
            dst.build_overviews([2, 4, 8, 16, 32], Resampling.average)
            dst.update_tags(ns='rio_overview', resampling='average')
        print('done')

    # ── Confirm output ────────────────────────────────────────────────────────
    with rasterio.open(REPROJECTED_TIF) as src:
        H, W = src.height, src.width
        REF_PROFILE = src.profile.copy()  # update for downstream steps
        new_res = src.res
        new_crs = src.crs

    print(f'\nReprojected file : {os.path.basename(REPROJECTED_TIF)}')
    print(f'CRS              : EPSG:{new_crs.to_epsg()}  ({pCRS.from_user_input(new_crs.to_wkt()).name})')
    print(f'Resolution       : {new_res[0]:.2f} x {new_res[1]:.2f} m')
    print(f'Dimensions       : {W} x {H} px  ({H*W/1e6:.1f} M pixels)')

    # ── Update MERGED_TIF to point to reprojected version ────────────────────
    # All downstream steps (Step 5 composite, Step 7 SAM) read from MERGED_TIF
    MERGED_TIF = REPROJECTED_TIF

    # Update band index remapping (unchanged — same bands, same order)
    merged_early_idx = BAND_REMAP[early_idx+1] - 1
    merged_peak_idx  = BAND_REMAP[peak_idx+1]  - 1
    merged_late_idx  = BAND_REMAP[late_idx+1]  - 1
    print(f'MERGED_TIF updated to reprojected file.')
    print(f'Steps 5, 7, 8 will use: {os.path.basename(MERGED_TIF)}')


---
## Step 5 — Build Phenological RGB Composite

Reads one band at a time to minimise RAM, stretches to uint8, writes directly
to `PHENO_RGB_TIF`. **Cached** — skip if file exists.

In [ ]:
from sklearn.decomposition import IncrementalPCA
from scipy.ndimage import uniform_filter

def stretch_channel(arr, method='percentile', p_low=2, p_high=98):
    """Stretch float32 array to uint8. Memory-efficient, float32 throughout."""
    valid = np.isfinite(arr)
    filled = arr.astype(np.float32)
    filled[~valid] = float(np.nanmedian(arr)) if valid.any() else 0.0
    if method == 'percentile':
        lo = float(np.percentile(filled[valid], p_low))  if valid.any() else 0.0
        hi = float(np.percentile(filled[valid], p_high)) if valid.any() else 1.0
        np.subtract(filled, lo,         out=filled)
        np.divide(  filled, hi-lo+1e-8, out=filled)
        np.clip(    filled, 0, 1,       out=filled)
    elif method == 'equalize':
        lo, hi = filled.min(), filled.max()
        np.subtract(filled, lo,         out=filled)
        np.divide(  filled, hi-lo+1e-8, out=filled)
        np.clip(filled, 0, 1, out=filled)
        filled = exposure.equalize_hist(filled).astype(np.float32)
    elif method == 'adaptive':
        lo, hi = filled.min(), filled.max()
        np.subtract(filled, lo,         out=filled)
        np.divide(  filled, hi-lo+1e-8, out=filled)
        np.clip(filled, 0, 1, out=filled)
        ks = max(64, min(256, arr.shape[0]//16, arr.shape[1]//16))
        filled = exposure.equalize_adapthist(
            filled, kernel_size=ks, clip_limit=0.03
        ).astype(np.float32)
    else:
        raise ValueError(f'Unknown method: {method}')
    out = (filled * 255).astype(np.uint8)
    out[~valid] = 0
    gc.collect()
    return out


# ── Set output filename based on mode ─────────────────────────────────────────
PHENO_RGB_TIF = os.path.join(OUTPUT_DIR, f'phenology_rgb_{COMPOSITE_MODE}.tif')

if os.path.exists(PHENO_RGB_TIF):
    print(f'Cached: {PHENO_RGB_TIF}')
    print('Delete to rebuild.')

elif COMPOSITE_MODE == 'tri_temporal':
    # ── Tri-temporal: read 3 bands, stretch, write ────────────────────────────
    print(f'Building tri_temporal composite ({STRETCH_METHOD})...  RAM:{ram_gb():.1f}GB')
    t0 = time.time()

    out_profile = REF_PROFILE.copy()
    out_profile.update(count=3, dtype=rasterio.uint8, compress='lzw',
                       photometric='RGB', nodata=0, driver='GTiff')

    with rasterio.open(PHENO_RGB_TIF, 'w', **out_profile) as dst:
        for out_band, (src_b, lbl) in enumerate(
            [(merged_early_idx+1, f'R ({el})'),
             (merged_peak_idx+1,  f'G ({pl})'),
             (merged_late_idx+1,  f'B ({ll})')],
            start=1
        ):
            print(f'  Band {out_band}/3: {lbl}  RAM:{ram_gb():.1f}GB...', end=' ')
            t_b = time.time()
            with rasterio.open(MERGED_TIF) as src:
                band = src.read(src_b).astype(np.float32)
            band[(band < -1) | (band > 1)] = np.nan
            stretched = stretch_channel(band, STRETCH_METHOD)
            dst.write(stretched, out_band)
            del band, stretched; gc.collect()
            print(f'{time.time()-t_b:.1f}s')

    print(f'Done: {time.time()-t0:.0f}s  {os.path.getsize(PHENO_RGB_TIF)/1e6:.0f} MB')

elif COMPOSITE_MODE == 'pca':
    # ── PCA composite: IncrementalPCA on monthly stack ────────────────────────
    print(f'Building PCA composite...  RAM:{ram_gb():.1f}GB')
    print(f'  Months : {[MONTH_LABELS[b-1] for b in BANDS_NEEDED]}')
    print(f'  R=PC{PCA_R_COMPONENT+1}  G=PC{PCA_G_COMPONENT+1}  B={B_CHANNEL_METHOD}')
    t0 = time.time()

    n_sel        = len(BANDS_NEEDED)
    n_components = max(PCA_R_COMPONENT, PCA_G_COMPONENT) + 1
    BATCH_ROWS   = max(64, min(256, H // 10))
    n_batches    = -(-H // BATCH_ROWS)  # ceiling division

    # ── Pass 1: compute per-band mean/std from downsampled sample ─────────────
    print(f'\n[1/4] Computing band statistics...')
    band_means, band_stds = [], []
    with rasterio.open(MERGED_TIF) as src:
        for i, src_band in enumerate(BANDS_NEEDED):
            samp = src.read(
                i + 1,
                out_shape=(min(500, H), min(500, W)),
                resampling=Resampling.average
            ).astype(np.float32)
            valid = samp[(samp > -1) & (samp < 1) & np.isfinite(samp)]
            band_means.append(float(valid.mean()) if len(valid) > 0 else 0.0)
            band_stds.append( float(valid.std())  if len(valid) > 0 else 1.0)
    band_means = np.array(band_means, dtype=np.float32)
    band_stds  = np.array(band_stds,  dtype=np.float32) + 1e-8
    print(f'  Band means: {band_means.round(3)}')

    # ── Pass 2: fit IncrementalPCA row-by-row ─────────────────────────────────
    print(f'\n[2/4] Fitting IncrementalPCA ({n_components} components, '
          f'{n_batches} batches)...')
    ipca = IncrementalPCA(n_components=n_components)

    with rasterio.open(MERGED_TIF) as src:
        for batch_i in range(n_batches):
            r0 = batch_i * BATCH_ROWS
            r1 = min(r0 + BATCH_ROWS, H)
            window = rasterio.windows.Window(0, r0, W, r1 - r0)
            batch = np.stack([
                src.read(i + 1, window=window).astype(np.float32)
                for i in range(n_sel)
            ], axis=0)  # (n_sel, rows, W)
            X = batch.reshape(n_sel, -1).T  # (rows*W, n_sel)
            del batch
            X = (X - band_means) / band_stds
            valid_rows = np.isfinite(X).all(axis=1)
            if valid_rows.sum() >= n_components:
                ipca.partial_fit(X[valid_rows])
            del X; gc.collect()
            if batch_i % max(1, n_batches // 5) == 0:
                print(f'  Batch {batch_i+1}/{n_batches} ({(batch_i+1)/n_batches*100:.0f}%)  '
                      f'RAM:{ram_gb():.1f}GB')

    print(f'  Explained variance:')
    for i, ev in enumerate(ipca.explained_variance_ratio_):
        bar = '█' * int(ev * 40)
        print(f'    PC{i+1}: {ev*100:5.1f}%  {bar}')

    # ── Pass 3: transform to PC scores row-by-row ─────────────────────────────
    print(f'\n[3/4] Transforming to PC scores...')
    pc_r = np.full((H, W), np.nan, dtype=np.float32)
    pc_g = np.full((H, W), np.nan, dtype=np.float32)

    with rasterio.open(MERGED_TIF) as src:
        for batch_i in range(n_batches):
            r0 = batch_i * BATCH_ROWS
            r1 = min(r0 + BATCH_ROWS, H)
            bh = r1 - r0
            window = rasterio.windows.Window(0, r0, W, bh)
            batch = np.stack([
                src.read(i + 1, window=window).astype(np.float32)
                for i in range(n_sel)
            ], axis=0)
            X = batch.reshape(n_sel, -1).T
            del batch
            X = (X - band_means) / band_stds
            valid_rows = np.isfinite(X).all(axis=1)
            scores = np.full((bh * W, n_components), np.nan, dtype=np.float32)
            if valid_rows.sum() > 0:
                scores[valid_rows] = ipca.transform(X[valid_rows]).astype(np.float32)
            del X
            pc_r[r0:r1] = scores[:, PCA_R_COMPONENT].reshape(bh, W)
            pc_g[r0:r1] = scores[:, PCA_G_COMPONENT].reshape(bh, W)
            del scores; gc.collect()
            if batch_i % max(1, n_batches // 5) == 0:
                print(f'  Batch {batch_i+1}/{n_batches}  RAM:{ram_gb():.1f}GB')

    # ── B channel: temporal CV or local contrast ──────────────────────────────
    print(f'\n[4/4] Computing B channel ({B_CHANNEL_METHOD})...')
    if B_CHANNEL_METHOD == 'cv':
        sum_arr   = np.zeros((H, W), dtype=np.float64)
        sum2_arr  = np.zeros((H, W), dtype=np.float64)
        count_arr = np.zeros((H, W), dtype=np.int16)
        with rasterio.open(MERGED_TIF) as src:
            for i in range(n_sel):
                band = src.read(i + 1).astype(np.float32)
                valid = (band > -1) & (band < 1) & np.isfinite(band)
                sum_arr[valid]   += band[valid]
                sum2_arr[valid]  += band[valid] ** 2
                count_arr[valid] += 1
                del band; gc.collect()
        with np.errstate(divide='ignore', invalid='ignore'):
            mean_arr = np.where(count_arr > 0, sum_arr / count_arr, np.nan)
            var_arr  = np.where(count_arr > 0, sum2_arr / count_arr - mean_arr**2, np.nan)
            b_raw    = np.sqrt(np.maximum(var_arr, 0)) / (mean_arr + 1e-6)
        del sum_arr, sum2_arr, count_arr, mean_arr, var_arr
    elif B_CHANNEL_METHOD == 'local_contrast':
        peak_idx_sel = int(np.nanargmax(band_means))
        with rasterio.open(MERGED_TIF) as src:
            peak_band = src.read(peak_idx_sel + 1).astype(np.float32)
        peak_band[(peak_band < -1) | (peak_band > 1)] = np.nan
        filled = np.nan_to_num(peak_band)
        local_mean = uniform_filter(filled, size=41)
        b_raw = peak_band - local_mean
        del filled, local_mean
    gc.collect()

    # ── Stretch and write ─────────────────────────────────────────────────────
    print(f'  Stretching ({STRETCH_METHOD})...')
    out_profile = REF_PROFILE.copy()
    out_profile.update(count=3, dtype=rasterio.uint8, compress='lzw',
                       photometric='RGB', nodata=0, driver='GTiff')

    with rasterio.open(PHENO_RGB_TIF, 'w', **out_profile) as dst:
        for out_band, (arr, lbl) in enumerate(
            [(pc_r, f'R=PC{PCA_R_COMPONENT+1}'),
             (pc_g, f'G=PC{PCA_G_COMPONENT+1}'),
             (b_raw, f'B={B_CHANNEL_METHOD}')],
            start=1
        ):
            print(f'  Band {out_band}/3: {lbl}  RAM:{ram_gb():.1f}GB...', end=' ')
            t_b = time.time()
            stretched = stretch_channel(arr, STRETCH_METHOD)
            dst.write(stretched, out_band)
            del stretched; gc.collect()
            print(f'{time.time()-t_b:.1f}s')

    del pc_r, pc_g, b_raw; gc.collect()
    print(f'Done: {time.time()-t0:.0f}s  {os.path.getsize(PHENO_RGB_TIF)/1e6:.0f} MB')

# ── Preview ───────────────────────────────────────────────────────────────────
with rasterio.open(PHENO_RGB_TIF) as src:
    s    = min(1.0, 700 / max(src.width, src.height))
    prev = src.read(out_shape=(3, int(src.height*s), int(src.width*s)),
                    resampling=Resampling.average)

if COMPOSITE_MODE == 'tri_temporal':
    title = f'Tri-temporal RGB  (R={el}, G={pl}, B={ll})  stretch={STRETCH_METHOD}'
else:
    ev_r = ipca.explained_variance_ratio_[PCA_R_COMPONENT] * 100
    ev_g = ipca.explained_variance_ratio_[PCA_G_COMPONENT] * 100
    title = (f'PCA composite  '
             f'R=PC{PCA_R_COMPONENT+1}({ev_r:.1f}%)  '
             f'G=PC{PCA_G_COMPONENT+1}({ev_g:.1f}%)  '
             f'B={B_CHANNEL_METHOD}  stretch={STRETCH_METHOD}')

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(np.moveaxis(prev, 0, -1))
ax.set_title(title, fontsize=10)
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'composite_preview.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Step 6 — Load SAM2

Loads SAM2 with the parameters defined in Step 2.
SAM2 weights (~180 MB) are downloaded from GitHub on first run.

In [ ]:
from samgeo import SamGeo2

sam = SamGeo2(
    model_id='sam2-hiera-large',
    device=DEVICE,
    apply_postprocessing=False,
    points_per_side=POINTS_PER_SIDE,
    pred_iou_thresh=PRED_IOU_THRESH,
    stability_score_thresh=STABILITY_THRESH,
    min_mask_region_area=MIN_MASK_AREA,
    crop_n_layers=CROP_N_LAYERS,
    crop_n_points_downscale_factor=CROP_N_POINTS_DOWNSCALE_FACTOR,
)
print('SAM2 loaded.')
print(f'  points_per_side  : {POINTS_PER_SIDE}')
print(f'  pred_iou_thresh  : {PRED_IOU_THRESH}')
print(f'  stability_thresh : {STABILITY_THRESH}')
print(f'  min_mask_area    : {MIN_MASK_AREA} px')
print(f'  crop_n_layers    : {CROP_N_LAYERS}')
print(f'  crop_n_points_downscale  : {CROP_N_POINTS_DOWNSCALE_FACTOR}')

---
## Step 7 — Run SAM Segmentation

Reads CRS and transform from the GeoTIFF header, then converts the composite
to a **memory-mapped numpy array** (`.dat` file) — the OS pages in only the
tiles SAM actually reads, keeping RAM flat for large scenes.

SAM runs on the memmap array directly (no file format overhead). The mask
raster is built in-memory, then polygonised in Step 8 with the saved CRS
and transform re-attached.

**Full scene mode** — processes the entire composite in one pass.

**Tiled mode** (`USE_TILING = True`) — slides a window across the composite,
saves each tile prediction to `PRED_DIR`. Per-tile caching means re-runs
skip tiles that already exist.

Both modes are **cached** — skip if output already exists.

In [ ]:
import math, os
from rasterio.windows import Window
import rasterio.windows

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OPENCV_LOG_LEVEL'] = 'ERROR'

t0 = time.time()

# ── Read CRS and transform — no pixels ───────────────────────────────────────
with rasterio.open(PHENO_RGB_TIF) as src:
    SAM_CRS       = src.crs
    SAM_TRANSFORM = src.transform
    SCENE_H       = src.height
    SCENE_W       = src.width
    N_BANDS       = src.count
    SCENE_PROFILE = src.profile.copy()

print(f'Scene     : {SCENE_W} x {SCENE_H} px  ({SCENE_H*SCENE_W/1e6:.1f} M px)')
print(f'CRS saved : EPSG:{SAM_CRS.to_epsg()}')
print(f'Transform : {SAM_TRANSFORM}')
print(f'RAM       : {ram_gb():.1f} GB available')

if not USE_TILING:
    # ── Full scene via memmap ─────────────────────────────────────────────────
    if os.path.exists(SAM_MASKS_TIF):
        print(f'\nCached: {SAM_MASKS_TIF}  (delete to re-run)')
    else:
        # ── Build memmap ──────────────────────────────────────────────────────
        if os.path.exists(MEMMAP_PATH):
            print(f'\nMemmap cached: {MEMMAP_PATH}')
        else:
            print(f'\nBuilding memmap {SCENE_W}x{SCENE_H}x{N_BANDS} uint8...')
            mm = np.memmap(MEMMAP_PATH, dtype=np.uint8, mode='w+',
                           shape=(SCENE_H, SCENE_W, N_BANDS))
            with rasterio.open(PHENO_RGB_TIF) as src:
                for b in range(N_BANDS):
                    print(f'  Band {b+1}/{N_BANDS}...', end=' ')
                    t_b = time.time()
                    mm[:, :, b] = src.read(b + 1)
                    mm.flush()
                    print(f'{time.time()-t_b:.1f}s  RAM:{ram_gb():.1f}GB')
            del mm; gc.collect()
            print(f'Memmap written: {os.path.getsize(MEMMAP_PATH)/1e6:.0f} MB')

        # ── Open memmap read-only (zero RAM cost) ─────────────────────────────
        print(f'\nOpening memmap read-only...')
        image_mm = np.memmap(MEMMAP_PATH, dtype=np.uint8, mode='r',
                             shape=(SCENE_H, SCENE_W, N_BANDS))
        print(f'Memmap shape : {image_mm.shape}  RAM:{ram_gb():.1f}GB')

        # ── Run SAM on memmap array ───────────────────────────────────────────
        print(f'\nRunning SAM2 on memmap array...')
        raw_masks = sam.mask_generator.generate(image_mm)
        print(f'SAM done: {len(raw_masks)} masks  RAM:{ram_gb():.1f}GB')
        del image_mm; gc.collect()

        # ── Build integer mask raster ─────────────────────────────────────────
        print('Building mask raster...')
        mask_raster = np.zeros((SCENE_H, SCENE_W), dtype=np.int32)
        for i, m in enumerate(raw_masks, start=1):
            mask_raster[m['segmentation']] = i
        del raw_masks; gc.collect()
        print(f'Mask raster: {mask_raster.max():,} segments')

        # ── Save mask raster — CRS and transform attached ─────────────────────
        mask_profile = SCENE_PROFILE.copy()
        mask_profile.update(count=1, dtype=rasterio.int32,
                            compress='lzw', nodata=0)
        with rasterio.open(SAM_MASKS_TIF, 'w', **mask_profile) as dst:
            dst.write(mask_raster, 1)
        del mask_raster; gc.collect()
        print(f'Saved: {SAM_MASKS_TIF}  ({os.path.getsize(SAM_MASKS_TIF)/1e6:.0f} MB)')

else:
    # ── Tiled mode ────────────────────────────────────────────────────────────
    # Each tile: read from composite, write to memmap tile, run SAM, save pred
    os.makedirs(PRED_DIR, exist_ok=True)
    stride = SAM_TILE_SIZE - TILE_OVERLAP
    xs = list(range(0, SCENE_W - SAM_TILE_SIZE, stride)) + [max(0, SCENE_W - SAM_TILE_SIZE)]
    ys = list(range(0, SCENE_H - SAM_TILE_SIZE, stride)) + [max(0, SCENE_H - SAM_TILE_SIZE)]
    origins = [(x, y) for y in sorted(set(ys)) for x in sorted(set(xs))]
    n_cached = sum(1 for x, y in origins
                   if os.path.exists(os.path.join(PRED_DIR, f'tile_{x:06d}_{y:06d}.tif')))
    print(f'\nTiled mode: {len(origins)} tiles  '
          f'({n_cached} cached, {len(origins)-n_cached} to run)')

    # Tile memmap — reused for each tile to avoid repeated allocation
    tile_mm_path = os.path.join(OUTPUT_DIR, '_tile_tmp.dat')

    with rasterio.open(PHENO_RGB_TIF) as scene_src:
        for idx, (x, y) in enumerate(origins):
            w = min(SAM_TILE_SIZE, SCENE_W - x)
            h = min(SAM_TILE_SIZE, SCENE_H - y)
            if w < 64 or h < 64:
                continue

            pred_path = os.path.join(PRED_DIR, f'tile_{x:06d}_{y:06d}.tif')
            if os.path.exists(pred_path):
                if idx % 20 == 0:
                    print(f'  [{idx+1}/{len(origins)}] cached')
                continue

            pct = (idx+1) / len(origins) * 100
            print(f'  [{idx+1}/{len(origins)}] ({pct:.0f}%) '
                  f'x={x} y={y} {w}x{h}  RAM:{ram_gb():.1f}GB...', end=' ')
            t_tile = time.time()

            # Read tile from composite into a memmap array
            window    = Window(x, y, w, h)
            tile_data = scene_src.read(window=window)  # (3, h, w) uint8
            tile_hwc  = np.moveaxis(tile_data, 0, -1).copy()  # (h, w, 3)
            del tile_data

            # Run SAM on tile array
            raw_masks = sam.mask_generator.generate(tile_hwc)
            del tile_hwc; gc.collect()

            # Build tile mask raster
            tile_mask = np.zeros((h, w), dtype=np.int32)
            for i, m in enumerate(raw_masks, start=1):
                tile_mask[m['segmentation']] = i
            del raw_masks; gc.collect()

            # Save tile prediction with correct geotransform
            tile_transform = rasterio.windows.transform(window, scene_src.transform)
            tile_profile = SCENE_PROFILE.copy()
            tile_profile.update(
                count=1, dtype=rasterio.int32,
                width=w, height=h,
                transform=tile_transform,
                compress='lzw', nodata=0
            )
            with rasterio.open(pred_path, 'w', **tile_profile) as dst:
                dst.write(tile_mask, 1)
            del tile_mask; gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            print(f'{time.time()-t_tile:.0f}s')

    # Clean up tile tmp file
    if os.path.exists(tile_mm_path):
        os.remove(tile_mm_path)

    n_done = sum(1 for x, y in origins
                 if os.path.exists(os.path.join(PRED_DIR, f'tile_{x:06d}_{y:06d}.tif')))
    print(f'\nTiling done: {n_done}/{len(origins)} tiles  {time.time()-t0:.0f}s')

print(f'\nStep 7 complete: {time.time()-t0:.0f}s')


---
## Step 8 — Vectorise

Polygonises the SAM mask raster using `rasterio.features.shapes`.
Re-attaches `SAM_CRS` and `SAM_TRANSFORM` saved in Step 7 so the output
GeoPackage is correctly georeferenced.

For tiled mode, processes one column of tiles at a time to keep RAM flat.

**Cached** — skip if output already exists.

In [ ]:
from collections import defaultdict
from shapely.geometry import shape

t0 = time.time()

if os.path.exists(SAM_VECTOR):
    print(f'Cached: {SAM_VECTOR}  (delete to re-run)')
    gdf = gpd.read_file(SAM_VECTOR)
    print(f'Loaded {len(gdf):,} segments.')

elif not USE_TILING:
    # ── Full scene vectorisation ──────────────────────────────────────────────
    print('Vectorising full scene mask...')
    # CRS and transform were saved in Step 7 — use directly
    with rasterio.open(SAM_MASKS_TIF) as src:
        mask  = src.read(1).astype(np.int32)
    trans = SAM_TRANSFORM
    crs   = SAM_CRS

    geoms = [
        {'geometry': shape(g), 'segment_id': int(v)}
        for g, v in rio_shapes(mask, mask=(mask>0).astype(np.uint8),
                                connectivity=4, transform=trans)
    ]
    del mask; gc.collect()

    gdf = gpd.GeoDataFrame(geoms, crs=crs)
    gdf = gdf.dissolve(by='segment_id').reset_index()
    gdf['geometry'] = gdf['geometry'].buffer(0)
    ea  = gdf.to_crs('EPSG:6933')
    gdf['area_ha'] = ea.geometry.area / 10000
    gdf.to_file(SAM_VECTOR, driver='GPKG')
    print(f'Done: {len(gdf):,} segments  {time.time()-t0:.0f}s')

else:
    # ── Tiled vectorisation — one column at a time ────────────────────────────
    preds = sorted([
        os.path.join(PRED_DIR, f)
        for f in os.listdir(PRED_DIR)
        if f.endswith('.tif') and not f.startswith('_')
    ])
    print(f'Vectorising {len(preds)} tile predictions...')

    # CRS saved in Step 7 — use directly
    tile_crs   = SAM_CRS
    pixel_size = abs(SAM_TRANSFORM.a)

    # Parse tile coords from filenames
    coords = []
    for p in preds:
        base = os.path.basename(p).replace('tile_','').replace('.tif','')
        pts  = base.split('_')
        if len(pts) == 2:
            try: coords.append((int(pts[0]), int(pts[1]), p))
            except ValueError: pass
    if not coords:
        coords = [(0, i*SAM_TILE_SIZE, p) for i, p in enumerate(preds)]

    cols = defaultdict(list)
    for x, y, p in coords:
        cols[x].append((y, p))
    col_xs = sorted(cols.keys())

    first_write = True
    total = 0

    for ci, x in enumerate(col_xs):
        tiles = sorted(cols[x])
        print(f'\nCol {ci+1}/{len(col_xs)} (x={x}, {len(tiles)} tiles)  RAM:{ram_gb():.1f}GB')
        frames = []

        for y, p in tiles:
            try:
                with rasterio.open(p) as src:
                    m  = src.read(1).astype(np.int32)
                    tr = src.transform
                    nd = src.nodata
                if nd is not None:
                    m[m == int(nd)] = 0
                geoms = [
                    {'geometry': shape(g), 'segment_id': int(v)}
                    for g, v in rio_shapes(
                        m, mask=(m>0).astype(np.uint8),
                        connectivity=4, transform=tr
                    )
                ]
                if geoms:
                    frames.append(gpd.GeoDataFrame(geoms, crs=tile_crs))
                    print(f'  y={y}: {len(geoms):,}')
                del m, geoms; gc.collect()
            except Exception as e:
                print(f'  y={y}: ERROR {e}')

        if not frames:
            continue

        col = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=tile_crs)
        del frames; gc.collect()
        col['geometry'] = col['geometry'].buffer(0)
        ea  = col.to_crs('EPSG:6933')
        col['area_ha'] = ea.geometry.area / 10000
        del ea; gc.collect()

        if first_write:
            col.to_file(SAM_VECTOR, driver='GPKG'); first_write = False
        else:
            col.to_file(SAM_VECTOR, driver='GPKG', mode='a')

        total += len(col)
        print(f'  Written {len(col):,}  total {total:,}  RAM:{ram_gb():.1f}GB')
        del col; gc.collect()

    gdf = gpd.read_file(SAM_VECTOR)
    print(f'\nDone: {len(gdf):,} segments  {time.time()-t0:.0f}s')

if 'area_ha' in gdf.columns:
    print(f'Area (ha): min={gdf.area_ha.min():.3f}  '
          f'median={gdf.area_ha.median():.2f}  '
          f'max={gdf.area_ha.max():.1f}')

---
## Step 9 — Plot Results

In [ ]:
# Load if not already in memory
if 'gdf' not in dir():
    gdf = gpd.read_file(SAM_VECTOR)
    print(f'Loaded {len(gdf):,} segments')

# Load composite preview
with rasterio.open(PHENO_RGB_TIF) as src:
    s = min(1.0, 700 / max(src.width, src.height))
    prev = src.read(out_shape=(3, int(src.height*s), int(src.width*s)),
                   resampling=Resampling.average)
    crs = src.crs
    ext = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

gdf_r = gdf.to_crs(crs)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left: composite only
axes[0].imshow(np.moveaxis(prev, 0, -1), extent=ext, aspect='equal')
axes[0].set_title(f'Phenological RGB  (R={el}, G={pl}, B={ll})', fontsize=11, fontweight='bold')
axes[0].axis('off')

# Right: composite + segment boundaries
axes[1].imshow(np.moveaxis(prev, 0, -1), extent=ext, aspect='equal')
gdf_r.boundary.plot(ax=axes[1], color='white', linewidth=0.5, alpha=0.9)
axes[1].set_title(f'SAM2 Segments — {len(gdf):,} objects', fontsize=11, fontweight='bold')
axes[1].axis('off')

plt.suptitle('Phenology Composite + SAM Segmentation', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'result.png'), dpi=150, bbox_inches='tight')
plt.show()

# Area distribution
if 'area_ha' in gdf.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    vmax = float(gdf.area_ha.quantile(0.98))
    gdf.plot(column='area_ha', ax=axes[0], cmap='YlOrRd',
             legend=True, vmin=0, vmax=vmax,
             legend_kwds={'label':'Area (ha)', 'shrink':0.6})
    axes[0].set_title(f'Area map  (clipped at {vmax:.1f} ha)', fontsize=11)
    axes[0].axis('off')

    axes[1].hist(gdf.area_ha.clip(upper=vmax), bins=50,
                 color='steelblue', edgecolor='white', linewidth=0.3)
    axes[1].axvline(float(gdf.area_ha.median()), color='red', linestyle='--',
                   label=f'Median: {gdf.area_ha.median():.2f} ha')
    axes[1].set_xlabel('Area (ha)'); axes[1].set_ylabel('Count')
    axes[1].set_title('Field size distribution'); axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'area_distribution.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
##### EXPORT 

import zipfile, os
import geopandas as gpd

# ── Load if not in memory ─────────────────────────────────────────────────────
if 'gdf' not in dir():
    gdf = gpd.read_file(SAM_VECTOR)

# ── Simplify in equal-area projection ────────────────────────────────────────
SIMPLIFY_TOLERANCE_M = 5.0  # metres — adjust to taste
                             # 5m  = light simplification, keeps field shape
                             # 10m = moderate, good for most uses
                             # 20m = aggressive, smaller file, rounded corners

print(f'Segments before simplify : {len(gdf):,}')
print(f'Simplify tolerance       : {SIMPLIFY_TOLERANCE_M} m')

# Project to equal-area for metric simplification
gdf_ea = gdf.to_crs('EPSG:6933')

# Simplify — preserve_topology=True prevents self-intersections
gdf_ea['geometry'] = gdf_ea['geometry'].simplify(
    SIMPLIFY_TOLERANCE_M,
    preserve_topology=True
)

# Drop empty/invalid geometries that can appear after simplification
gdf_ea = gdf_ea[gdf_ea.geometry.is_valid & ~gdf_ea.geometry.is_empty]
print(f'Segments after simplify  : {len(gdf_ea):,}')

# ── Reproject to WGS84 for KML/KMZ ───────────────────────────────────────────
gdf_wgs = gdf_ea.to_crs('EPSG:4326')

# ── Export as KMZ ────────────────────────────────────────────────────────────
kml_path = os.path.join(OUTPUT_DIR, 'sam_segments_simplified.kml')
kmz_path = os.path.join(OUTPUT_DIR, 'sam_segments_simplified.kmz')
gpkg_simplified = os.path.join(OUTPUT_DIR, 'sam_segments_simplified.gpkg')

# Save simplified GeoPackage (in original CRS)
gdf_ea.to_crs(gdf.crs).to_file(gpkg_simplified, driver='GPKG')

# Save KMZ
gdf_wgs.to_file(kml_path, driver='KML')
with zipfile.ZipFile(kmz_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(kml_path, 'sam_segments_simplified.kml')
os.remove(kml_path)

# File size comparison
orig_mb = os.path.getsize(SAM_VECTOR) / 1e6
kmz_mb  = os.path.getsize(kmz_path)   / 1e6
gpkg_mb = os.path.getsize(gpkg_simplified) / 1e6
print(f'\nFile sizes:')
print(f'  Original GeoPackage   : {orig_mb:.1f} MB')
print(f'  Simplified GeoPackage : {gpkg_mb:.1f} MB  ({gpkg_mb/orig_mb*100:.0f}% of original)')
print(f'  Simplified KMZ        : {kmz_mb:.1f} MB')